# **VoxShield - Pipeline Execution Notebook**
This notebook coordinates the complete training and evaluation pipeline for **VoxShield**, a dual-branch attention-based audio deepfake detection system.

### **Important: Choose Your Runtime Environment**

#### **Option A: Running on Local Runtime (Recommended)**
* Use this if you are connecting Google Colab to your **local Jupyter server** (where your GPU and local files are located).
* Do **NOT** run the Google Drive mount code.
* Navigating to your local project directory works out-of-the-box.

#### **Option B: Running on Google Colab Cloud Runtime**
* Use this if you are using Google's hosted cloud GPU.
* Since Google Colab Cloud cannot see your local `D:\` drive directly, you **MUST** first upload your `Dataset` and `VoxShield_Development` folders to your Google Drive.
* Once uploaded, run the Google Drive mount code.

## **Step 1: Environment and Path Setup**
Run the cells below depending on whether you are using a **Local Runtime** or **Cloud Runtime**.

In [4]:
# RUN THIS ONLY IF USING GOOGLE COLAB CLOUD RUNTIME (Option B)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted successfully.")
except ImportError:
    print("Not running in Colab Cloud. Skipping Drive mount.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted successfully.


In [5]:
# Install dependencies in your active Python environment
!pip install torch torchaudio transformers soundfile librosa pandas scikit-learn scipy matplotlib

In [ ]:
# RUN THIS TO SET UP YOUR PROJECT PATH (Supports both Local and Colab Cloud)
import os
if os.path.exists('/content'):
    print('[Environment] Running on Google Colab Cloud.')
    # Dynamically locate and navigate to the project directory in Google Drive
    def find_and_cd():
        candidates = [
            '/content/drive/MyDrive/project phase 1/VoxShield/VoxShield_Development',
            '/content/drive/MyDrive/Project Work phase 1/VoxShield/VoxShield_Development',
            '/content/drive/MyDrive/VoxShield/VoxShield_Development',
            '/content/drive/MyDrive/project phase 1/VoxShield_Development',
        ]
        for c in candidates:
            if os.path.exists(c):
                print(f'Found project development folder at: {c}')
                get_ipython().run_line_magic('cd', c)
                return True
        print('Searching for VoxShield_Development in Google Drive...')
        for root, dirs, files in os.walk('/content/drive/MyDrive'):
            depth = root.replace('/content/drive/MyDrive', '').count(os.sep)
            if depth > 3: continue
            if 'VoxShield_Development' in dirs:
                target = os.path.join(root, 'VoxShield_Development')
                print(f'Found project development folder at: {target}')
                get_ipython().run_line_magic('cd', target)
                return True
        print('[Error] Could not automatically find VoxShield_Development in Google Drive.')
        print('Please run manually: %cd "/content/drive/MyDrive/your-path/VoxShield_Development"')
        return False
    find_and_cd()
else:
    print('[Environment] Running on Local Machine.')
    local_path = 'D:/Project Work phase 1/VoxShield/VoxShield_Development'
    if os.path.exists(local_path):
        get_ipython().run_line_magic('cd', local_path)
        print(f'Navigated to local project folder: {local_path}')
    else:
        print(f'[Warning] Local path not found: {local_path}')

if os.path.exists('/content'):
    get_ipython().system('ls')
else:
    get_ipython().system('dir')


## **Step 2: Generate Dataset Manifests (Phase 2)**
This script parses the protocol files for ASVspoof 2019 and ASVspoof 2021 logical access trials and writes standard CSV manifest lists.

In [ ]:
# Run manifest builder script
!python manifest_builder.py

## **Step 3: Preprocessing and Dataloader Sanity Checks (Phase 3 & 4)**
Verify the dataset class loading, resampling, repeat padding, and default feature extraction (LFCC) shapes.

In [ ]:
!python dataset.py

## **Step 4: Train and Evaluate Baseline Models (Phase 5)**
Train both frozen wav2vec2-base (Baseline A) and LFCC-CNN (Baseline B) classifier pipelines to establish comparison results.

In [ ]:
# Train Baseline A (wav2vec2 + MLP classifier)
!python baselines/baseline_wav2vec2.py

In [ ]:
# Train Baseline B (LFCC + 2D CNN classifier)
!python baselines/baseline_cnn.py

## **Step 5: Train VoxShield Core Architecture (Phase 6)**
This script trains the twin-branch fused VoxShield architecture using a two-stage scheduler (initially frozen wav2vec2, followed by unfreezing of the top 2 layers with a low learning rate).

In [ ]:
!python train.py

## **Step 6: Logit Calibration (Phase 7)**
Compute optimal temperature scaling on the validation dataset (dev_2019) to calibrate probability outputs.

In [ ]:
!python calibrate.py

## **Step 7: Inference / Prediction (Phase 7)**
Run prediction on any target flac audio file. The script outputs the calibrated spoof probability, the associated risk band (Green, Amber, Red), and highlights the most suspicious segment timeframes.

In [ ]:
# Run inference on a sample dev file (update filepath to test other files)
!python predict.py --audio_path "../Dataset/LA/ASVspoof2019_LA_dev/flac/LA_D_1048473.flac"